# Denoising Autoencoder and Variational Autoencoder (VAE)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Data Loading


In [ ]:
transform = transforms.Compose([
    # TODO: Transform to tensor
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = None # TODO
test_loader = None  # TODO

def add_noise(imgs, noise_level=0.5):
    return torch.clamp(imgs + noise_level * torch.randn_like(imgs), 0., 1.)


## Denoising Autoencoder: Model Definition

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            # TODO: conv2d (output=32) -> relu -> max pool -> conv2d (output=64) -> relu -> max pool
        )
        self.decoder = nn.Sequential(
            # TODO: conv transpose (output=32) -> relu -> conv transpose (output=1) -> sigmoid
        )

    def forward(self, x):
        return None   # TODO

### Training Loop

In [ ]:
model = None  # TODO
optimizer = None  # TODO
criterion = None  # TODO: use MSE and add parameter reduction='sum'

for epoch in range(10):
    # TODO: move model to right mode
    total_loss = None   # TODO: reset total loss count
    for imgs, _ in train_loader:
        imgs = None # TODO: move to right device
        noisy_imgs = None # TODO

        outputs = None    # TODO
        loss = None # TODO

        # TODO

        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")


### Reconstruction Visualization


In [ ]:
model.eval()
imgs, _ = next(iter(test_loader))
noisy_imgs = add_noise(imgs)

with torch.no_grad():
    recon = model(noisy_imgs.to(device)).cpu()

fig, axes = plt.subplots(3, 8, figsize=(12, 4))
for i in range(8):
    axes[0][i].imshow(imgs[i][0], cmap='gray'); axes[0][i].axis('off')
    axes[1][i].imshow(noisy_imgs[i][0], cmap='gray'); axes[1][i].axis('off')
    axes[2][i].imshow(recon[i][0], cmap='gray'); axes[2][i].axis('off')

axes[0][0].set_ylabel("Clean")
axes[1][0].set_ylabel("Noisy")
axes[2][0].set_ylabel("Recon")
plt.tight_layout()
plt.show()


## Variational Autoencoder (VAE)

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            # TODO: conv2d (64 output channels) -> relu -> conv2d (128 output channels) -> relu
        )
        self.fc_mu = None # TODO: Linear layer (128*7*7 inputs, 20 outputs)
        self.fc_logvar = None # TODO: same as above
        self.decoder_fc = None  # TODO: nn.Linear(20, 128*7*7)

        self.decoder = nn.Sequential(
            # TODO: transposed conv2d (128->64, 4, 2, 1) -> relu -> transposed conv2d (128->64, 4, 2, 1) -> sigmoid
        )

    # Sampling is a random operation: we move the randomness inside an external variable (eps) so that
    # gradients can flow through mu and logvar, which are treated as trainable parameters
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def forward(self, x):
        enc = self.encoder(x).view(x.size(0), -1)
        mu = None # TODO: apply self.fc_mu to enc
        logvar = None #TODO: apply self.fc_logvar to enc
        z = None  # TODO: reparametrize

        dec_input = self.decoder_fc(z).view(x.size(0), 128, 7, 7)

        return None, None, None # TODO: return decoder's output applied over dec_input, mu, and logvar


### Training Loop


In [ ]:
def kl_divergence(mu, logvar):
    """
    Computes the KL divergence between N(mu, sigma^2) and N(0, 1)

    Args:
        mu (Tensor): Mean of latent distribution (batch_size, latent_dim)
        logvar (Tensor): Log-variance (log(sigma^2)) of latent distribution

    Returns:
        Tensor: Scalar KL divergence (summed over batch)
    """
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())


In [ ]:
vae = None  # TODO
optimizer = None  # TODO
BCE= None # TODO: use Binary Cross Entropy Loss (use reduction='sum')

for epoch in range(20):
    # TODO
    total_loss = 0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        recon, mu, logvar = None  # TODO

        loss = None # TODO: sum of BCE and KL

        # TODO

        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")


### New Data Generation


In [ ]:
vae.eval()
with torch.no_grad():
    z = torch.randn(64, 20).to(device)
    dec = vae.decoder_fc(z).view(64, 128, 7, 7)
    gen_imgs = vae.decoder(dec).cpu()

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i in range(64):
    ax = axes[i//8][i%8]
    ax.imshow(gen_imgs[i][0], cmap='gray')
    ax.axis('off')
plt.suptitle("Generated Digits from VAE")
plt.show()
